# CUDA GNN inference — Colab runner

This notebook is self-contained. It builds the repository on Colab's local disk, stores benchmark results in Google Drive, validates all compiled backends, and runs a small synthetic GCN/GraphSAGE experiment.

Before running it, select **Runtime → Change runtime type → T4 GPU** (or another NVIDIA GPU). The public-dataset and million-node experiments near the end are opt-in.

In [ ]:
import shutil
import subprocess
import sys

for tool in ('git', 'g++', 'nvcc', 'nvidia-smi'):
    if shutil.which(tool) is None:
        raise RuntimeError(f'{tool} is unavailable. Select a Colab GPU runtime and reconnect.')

subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['nvcc', '--version'], check=True)

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT = Path('/content/cuda-gnn-inference')
RESULTS = Path('/content/drive/MyDrive/cuda-gnn-inference-results')
RESULTS.mkdir(parents=True, exist_ok=True)
print('Persistent results:', RESULTS)

In [ ]:
REPOSITORY = 'https://github.com/Alby02/cuda-gnn-inference.git'
BRANCH = 'fix/python-colab-synthetic-workflow'

if (PROJECT / '.git').is_dir():
    status = subprocess.run(
        ['git', '-C', str(PROJECT), 'status', '--porcelain'],
        check=True, text=True, capture_output=True,
    ).stdout.strip()
    if status:
        raise RuntimeError(f'{PROJECT} has local changes; restart the runtime or clean that checkout.')
    subprocess.run(['git', '-C', str(PROJECT), 'fetch', '--depth', '1', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
elif PROJECT.exists():
    raise RuntimeError(f'{PROJECT} exists but is not a Git checkout; restart the runtime or rename it.')
else:
    subprocess.run([
        'git', 'clone', '--depth', '1', '--branch', BRANCH, '--single-branch',
        REPOSITORY, str(PROJECT),
    ], check=True)
print('Using commit:', subprocess.run(
    ['git', '-C', str(PROJECT), 'rev-parse', '--short', 'HEAD'],
    check=True, text=True, capture_output=True,
).stdout.strip())

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'meson', 'ninja', 'pandas', '-r', str(PROJECT / 'python_libraries.txt'),
], check=True)

In [ ]:
build = Path('/content/cuda-gnn-build')
if (build / 'meson-private').is_dir():
    configure = ['meson', 'setup', '--wipe', str(build), str(PROJECT), '-Dopenmp=enabled', '-Dcuda=enabled']
else:
    configure = ['meson', 'setup', str(build), str(PROJECT), '-Dopenmp=enabled', '-Dcuda=enabled']
subprocess.run(configure, check=True)
subprocess.run(['meson', 'compile', '-C', str(build)], check=True)
EXECUTABLE = build / 'gnn'
help_result = subprocess.run([str(EXECUTABLE), '--help'], check=True, text=True, capture_output=True)
print(help_result.stdout)
for required in ('sequential', 'parallel', 'cuda'):
    if required not in help_result.stdout:
        raise RuntimeError(f'Expected backend {required!r} was not compiled')

## Correctness smoke tests

The first test compares the built-in demo across sequential, OpenMP, and CUDA. The second generates tracked fixtures and checks empty, isolated, undirected, weighted, GraphSAGE, GCN, and no-bias paths against the independent Python/PyG reference.

In [ ]:
import re
import numpy as np

def demo_output(backend):
    result = subprocess.run(
        [str(EXECUTABLE), '--backend', backend, '--warmups', '0', '--repetitions', '1'],
        check=True, text=True, capture_output=True, cwd=PROJECT,
    )
    rows = []
    for line in result.stdout.splitlines():
        match = re.search(r'\[([^\]]+)\]', line)
        if match:
            rows.append([float(value) for value in match.group(1).split(',')])
    if not rows:
        raise RuntimeError(f'{backend} produced no parseable output')
    return np.asarray(rows, dtype=np.float32)

sequential = demo_output('sequential')
for backend in ('parallel', 'cuda'):
    np.testing.assert_allclose(demo_output(backend), sequential, rtol=1e-5, atol=1e-5)
print('All native backends match:', sequential.tolist())

In [ ]:
subprocess.run([
    sys.executable, str(PROJECT / 'scripts/check_edge_cases.py'),
    '--native', str(EXECUTABLE),
    '--work-dir', '/content/gnn-edge-cases',
    '--backend', 'sequential', 'parallel', 'cuda',
], check=True, cwd=PROJECT)

## Small synthetic end-to-end benchmark

This exercises synthetic graph generation, model export, all native backends, PyTorch Geometric comparison, repeatability checks, CSV output, and plots. It varies each workload axis independently so the scaling plots contain real trends instead of duplicated single points.

In [ ]:
QUICK_RESULTS = RESULTS / 'quick-synthetic'
quick_command = [
    sys.executable, str(PROJECT / 'scripts/run_experiments.py'),
    '--native', str(EXECUTABLE),
    '--dataset', 'none',
    '--nodes', '512', '2048',
    '--widths', '16', '64',
    '--depths', '1', '3',
    '--skews', '0', '1',
    '--backend', 'sequential', 'parallel', 'cuda',
    '--threads', '1', '2',
    '--block-size', '128', '256',
    '--warmups', '1',
    '--repetitions', '3',
    '--repeat-checks', '1',
    '--output-dir', str(QUICK_RESULTS),
]
quick_result = subprocess.run(
    quick_command, cwd=PROJECT, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
print(quick_result.stdout)
if quick_result.returncode:
    print('\nNon-empty experiment error logs:')
    for log in sorted(QUICK_RESULTS.rglob('*.stderr.txt')):
        contents = log.read_text(encoding='utf-8', errors='replace').strip()
        if contents:
            print(f'\n--- {log.relative_to(QUICK_RESULTS)} ---\n{contents[-4000:]}')
    raise RuntimeError(f'Synthetic experiment failed with exit code {quick_result.returncode}')

In [ ]:
import pandas as pd
from IPython.display import Image, display

comparison = pd.read_csv(QUICK_RESULTS / 'comparison.csv')
display(comparison[[
    'workload', 'model_types', 'native_backend', 'framework_device',
    'threads', 'block_size', 'verification', 'max_abs_error',
    'native_mean_ms', 'framework_mean_ms', 'native_speedup',
]])
for plot in sorted((QUICK_RESULTS / 'plots').glob('*.png')):
    print(plot.name)
    display(Image(filename=str(plot), width=1100))

## Optional long experiments

Enable one experiment at a time and rerun the next cell. `RUN_SCALING` performs a genuine one-variable-at-a-time sweep. `REBUILD_EXISTING_PLOTS` upgrades figures already stored in Drive without rerunning inference. The million-node case can consume several gigabytes of host/GPU memory and substantial Colab time; it omits the sequential benchmark but still validates against PyTorch Geometric.

In [ ]:
RUN_CORA = False
RUN_SCALING = False
RUN_MILLION_NODES = False
REBUILD_EXISTING_PLOTS = True

def run_optional(name, arguments):
    output = RESULTS / name
    command = [
        sys.executable, str(PROJECT / 'scripts/run_experiments.py'),
        '--native', str(EXECUTABLE),
        *arguments,
        '--output-dir', str(output),
    ]
    print(f'\n=== Starting {name} ===', flush=True)
    process = subprocess.Popen(
        command, cwd=PROJECT, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code:
        raise RuntimeError(f'{name} failed with exit code {return_code}')
    print(f'=== Completed {name}: {output} ===', flush=True)
    return output

completed = {}
if RUN_CORA:
    completed['cora'] = run_optional('cora', [
        '--dataset', 'Cora', '--skip-synthetic',
        '--backend', 'sequential', 'parallel', 'cuda',
        '--threads', '1', '2', '--block-size', '128', '256',
        '--warmups', '2', '--repetitions', '10',
    ])

if RUN_SCALING:
    completed['scaling'] = run_optional('scaling', [
        '--dataset', 'none', '--nodes', '1000', '10000', '100000',
        '--widths', '32', '128', '--depths', '2', '4', '--skews', '0', '1',
        '--backend', 'sequential', 'parallel', 'cuda',
        '--threads', '1', '2', '--block-size', '128', '256',
        '--warmups', '2', '--repetitions', '5', '--repeat-checks', '1',
    ])

if RUN_MILLION_NODES:
    completed['million-nodes'] = run_optional('million-nodes', [
        '--dataset', 'none', '--nodes', '1000000', '--widths', '32',
        '--depths', '2', '--skews', '0',
        '--backend', 'parallel', 'cuda', '--threads', '2',
        '--block-size', '128', '256', '--warmups', '1',
        '--repetitions', '5', '--repeat-checks', '0',
    ])

if REBUILD_EXISTING_PLOTS:
    for name in ('cora', 'scaling', 'million-nodes'):
        output = RESULTS / name
        required = [output / file for file in ('workloads.json', 'samples.csv', 'comparison.csv')]
        if all(path.exists() for path in required):
            print(f'\n=== Rebuilding {name} plots from saved results ===', flush=True)
            subprocess.run([
                sys.executable, str(PROJECT / 'scripts/run_experiments.py'),
                '--plots-only', '--output-dir', str(output),
            ], check=True, cwd=PROJECT)
            completed.setdefault(name, output)

if not completed:
    print('Optional experiments skipped. Set one flag above to True to run it.')
else:
    import pandas as pd
    from IPython.display import Image, display
    for name, output in completed.items():
        print(f'\n=== {name} summary ===')
        comparison = pd.read_csv(output / 'comparison.csv')
        display(comparison[[
            'workload', 'model_types', 'native_backend', 'framework_device',
            'threads', 'block_size', 'verification', 'max_abs_error',
            'native_mean_ms', 'framework_mean_ms', 'native_speedup',
        ]])
        for plot in sorted((output / 'plots').glob('*.png')):
            print(plot.name)
            display(Image(filename=str(plot), width=1100))